# MNPS Job Classification Likelihood Scorer v8.0 - ULTIMATE ENHANCED EDITION

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/yourrepo/blob/main/MNPS_Job_Classification_v8_Ultimate_Enhanced.ipynb)

## 🚀 v8.0: The Complete Integration with Enhanced Evaluation

This ultimate version combines:
- **v7 Complete Integration**: Full feature set with comprehensive analysis
- **v4 KSAC-AWARE**: TF-IDF vectorization for competency matching
- **v2 Evaluation Framework**: Confusion matrices and ground truth comparison

### ✨ Complete Feature Set:

#### Core Analysis Engine
1. **🔬 Dual KSAC Similarity** - Both embedding cosine AND TF-IDF approaches
2. **💰 Advanced Cost Analysis** - Salary-weighted with tier multipliers
3. **⏱️ Dynamic Time Estimation** - Severity-adjusted correction hours
4. **🎯 Multi-Algorithm Scoring** - Ensemble approach for robustness

#### Enhanced Analytics
5. **📊 12-Panel Visualizations** - Extended dashboard with evaluation metrics
6. **⚠️ Advanced Borderline Detection** - Multi-factor edge case identification
7. **💡 Intelligent JD Suggestions** - Context-aware recommendations
8. **📈 Comprehensive Performance Metrics** - Including confusion analysis

#### Quality & Evaluation
9. **✅ Enhanced Confidence Engine** - 6-component assessment
10. **🎭 Pattern Recognition** - Historical performance tracking
11. **📉 Dynamic Calibration** - Self-adjusting thresholds
12. **🔍 Confusion Matrix Analysis** - Severity-weighted evaluation

#### New in v8.0
13. **📐 TF-IDF KSAC Analysis** - From v4's approach
14. **🎯 Ground Truth Comparison** - From v2's evaluation
15. **📊 Confusion Matrices** - Standard and severity-weighted
16. **🔒 Real Data Enforcement** - No synthetic fallback

### 📁 Required Files (NO SYNTHETIC DATA):
1. **Evaluation Resources.zip** - MUST contain: KSACs.csv, Salary_Data.csv, Time_to_Correct.csv, Role_Groups.csv
2. **Sample_JDs.csv** - Job descriptions file
3. **Job_Classifications_Batch.csv** - Classification results file

---

In [ ]:
#@title 1️⃣ Setup Environment and Mount Drive { display-mode: "form" }

from google.colab import drive, files
import os
import datetime
import zipfile
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report
import json
import io
import re
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
print("🔗 Mounting Google Drive...")
drive.mount('/content/drive')

# Create directory structure
base_path = '/content/drive/MyDrive/MNPS_Likelihood_Analysis_v8'
resources_path = os.path.join(base_path, 'Resources')
results_path = os.path.join(base_path, 'Results')
inputs_path = os.path.join(base_path, 'Inputs')
evaluation_path = os.path.join(base_path, 'Evaluation')

for path in [base_path, resources_path, results_path, inputs_path, evaluation_path]:
    os.makedirs(path, exist_ok=True)

# Create timestamped results folder
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
run_results_path = os.path.join(results_path, f'Run_v8_{timestamp}')
os.makedirs(run_results_path, exist_ok=True)

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')

print(f"✅ Environment ready!")
print(f"📁 Results will be saved to: {run_results_path}")
print(f"\n⚠️ IMPORTANT: This version requires real data - no synthetic fallback!")

In [ ]:
#@title 2️⃣ Configure Complete Parameters { display-mode: "form" }

# Human Baseline
HUMAN_BASELINE_MIN = 88  #@param {type:"number"}
HUMAN_BASELINE_TYPICAL = 91  #@param {type:"number"}
HUMAN_BASELINE_MAX = 94  #@param {type:"number"}

# Error Cost Weights
KSAC_DISSIMILARITY_WEIGHT = 0.40  #@param {type:"slider", min:0.2, max:0.6, step:0.05}
SALARY_IMPACT_WEIGHT = 0.35  #@param {type:"slider", min:0.2, max:0.5, step:0.05}
CORRECTION_TIME_WEIGHT = 0.25  #@param {type:"slider", min:0.1, max:0.4, step:0.05}

# KSAC Analysis Method
KSAC_METHOD = "Both (Ensemble)"  #@param ["Cosine Similarity", "TF-IDF", "Both (Ensemble)"]
TFIDF_NGRAM_RANGE = (1, 2)  #@param {type:"raw"}

# Confidence Thresholds
CONFIDENCE_THRESHOLD = 0.6  #@param {type:"slider", min:0.4, max:0.8, step:0.05}
CONFIDENCE_HIGH = 0.8  #@param {type:"slider", min:0.7, max:0.9, step:0.05}

# Feature Toggles
ENABLE_BORDERLINE_DETECTION = True  #@param {type:"boolean"}
ENABLE_JD_SUGGESTIONS = True  #@param {type:"boolean"}
ENABLE_PATTERN_DETECTION = True  #@param {type:"boolean"}
ENABLE_DYNAMIC_CALIBRATION = True  #@param {type:"boolean"}
ENABLE_CONFUSION_ANALYSIS = True  #@param {type:"boolean"}
ENABLE_TFIDF_ANALYSIS = True  #@param {type:"boolean"}

# Error Severity Thresholds
CRITICAL_ERROR_THRESHOLD = 0.75  #@param {type:"slider", min:0.6, max:0.9, step:0.05}
MAJOR_ERROR_THRESHOLD = 0.50  #@param {type:"slider", min:0.4, max:0.7, step:0.05}
MINOR_ERROR_THRESHOLD = 0.25  #@param {type:"slider", min:0.1, max:0.4, step:0.05}

# Correction Time Settings
BASE_CORRECTION_TIME = 8  #@param {type:"number"}
CRITICAL_TIME_MULTIPLIER = 10  #@param {type:"number"}
MAJOR_TIME_MULTIPLIER = 5  #@param {type:"number"}
MINOR_TIME_MULTIPLIER = 2  #@param {type:"number"}

# Evaluation Settings (from v2)
SEVERITY_SALARY_WEIGHT = 0.7  #@param {type:"slider", min:0.3, max:0.9, step:0.1}
SEVERITY_TIME_WEIGHT = 0.3  #@param {type:"slider", min:0.1, max:0.7, step:0.1}
INCLUDE_GROUND_TRUTH = False  #@param {type:"boolean"}

print("📊 Configuration Summary:")
print(f"  Human Baseline: {HUMAN_BASELINE_MIN}-{HUMAN_BASELINE_MAX}% (typical: {HUMAN_BASELINE_TYPICAL}%)")
print(f"  KSAC Method: {KSAC_METHOD}")
print(f"  Error Weights: KSAC={KSAC_DISSIMILARITY_WEIGHT}, Salary={SALARY_IMPACT_WEIGHT}, Time={CORRECTION_TIME_WEIGHT}")
print(f"  Confidence: Threshold={CONFIDENCE_THRESHOLD}, High={CONFIDENCE_HIGH}")
print(f"  Features Enabled:")
for feature, enabled in [
    ('Borderline Detection', ENABLE_BORDERLINE_DETECTION),
    ('JD Suggestions', ENABLE_JD_SUGGESTIONS),
    ('Pattern Detection', ENABLE_PATTERN_DETECTION),
    ('Dynamic Calibration', ENABLE_DYNAMIC_CALIBRATION),
    ('Confusion Analysis', ENABLE_CONFUSION_ANALYSIS),
    ('TF-IDF Analysis', ENABLE_TFIDF_ANALYSIS)
]:
    print(f"    - {feature}: {'✓' if enabled else '✗'}")

In [ ]:
#@title 3️⃣ Upload and Validate Required Files { display-mode: "form" }

print("📤 Please upload the following REQUIRED files:")
print("\n1. Evaluation Resources.zip")
print("   Must contain: KSACs.csv, Salary_Data.csv, Time_to_Correct.csv, Role_Groups.csv")
print("\n2. Sample_JDs.csv")
print("   Your job descriptions file")
print("\n3. Job_Classifications_Batch.csv")
print("   Your classification results file")
print("\n" + "="*60 + "\n")

uploaded = files.upload()

# Track what was uploaded
uploaded_files = list(uploaded.keys())
resources_loaded = False
jds_loaded = False
classifications_loaded = False

# Process uploaded files
for filename in uploaded_files:
    print(f"\n📁 Processing: {filename}")
    
    if 'evaluation' in filename.lower() and filename.endswith('.zip'):
        # Extract resources
        with zipfile.ZipFile(io.BytesIO(uploaded[filename]), 'r') as zip_ref:
            zip_ref.extractall('/content/temp_extract')
        
        # Move to resources folder
        for root, dirs, files_list in os.walk('/content/temp_extract'):
            for file in files_list:
                src = os.path.join(root, file)
                dst = os.path.join(resources_path, file)
                shutil.copy(src, dst)
                print(f"   ✅ Extracted: {file}")
        
        shutil.rmtree('/content/temp_extract', ignore_errors=True)
        resources_loaded = True
    
    elif 'sample' in filename.lower() and 'jd' in filename.lower():
        # Save job descriptions
        df = pd.read_csv(io.BytesIO(uploaded[filename]))
        df.to_csv(os.path.join(inputs_path, 'Sample_JDs.csv'), index=False)
        job_descriptions = df
        jds_loaded = True
        print(f"   ✅ Loaded {len(df)} job descriptions")
    
    elif 'classification' in filename.lower() and 'batch' in filename.lower():
        # Save classifications
        df = pd.read_csv(io.BytesIO(uploaded[filename]))
        df.to_csv(os.path.join(inputs_path, 'Job_Classifications_Batch.csv'), index=False)
        classifications = df
        classifications_loaded = True
        print(f"   ✅ Loaded {len(df)} classifications")

# Validate all required files are present
print("\n" + "="*60)
print("📋 Validation Report:")
print("="*60)

validation_passed = True

# Check resources
required_resources = ['KSACs.csv', 'Salary_Data.csv', 'Time_to_Correct.csv', 'Role_Groups.csv']
missing_resources = []

for resource in required_resources:
    resource_path = os.path.join(resources_path, resource)
    if os.path.exists(resource_path):
        print(f"  ✅ {resource} found")
    else:
        print(f"  ❌ {resource} MISSING")
        missing_resources.append(resource)
        validation_passed = False

# Check input files
if jds_loaded:
    print(f"  ✅ Sample_JDs.csv loaded ({len(job_descriptions)} records)")
else:
    print(f"  ❌ Sample_JDs.csv MISSING")
    validation_passed = False

if classifications_loaded:
    print(f"  ✅ Job_Classifications_Batch.csv loaded ({len(classifications)} records)")
else:
    print(f"  ❌ Job_Classifications_Batch.csv MISSING")
    validation_passed = False

# Final validation
if not validation_passed:
    print("\n⚠️ ERROR: Required files missing!")
    print("Please upload all required files and run this cell again.")
    if missing_resources:
        print(f"\nMissing from Evaluation Resources.zip: {', '.join(missing_resources)}")
    raise ValueError("Required files not found. Cannot proceed without real data.")
else:
    print("\n✅ All required files validated successfully!")
    print("\nProceeding with REAL DATA analysis...")

In [ ]:
#@title 4️⃣ Load and Prepare All Data { display-mode: "form" }

print("📂 Loading resource files...")

# Load KSACs
ksac_df = pd.read_csv(os.path.join(resources_path, 'KSACs.csv'))
print(f"✅ KSACs: {len(ksac_df)} records")

# Load Salary Data
salary_df = pd.read_csv(os.path.join(resources_path, 'Salary_Data.csv'))
print(f"✅ Salary Data: {len(salary_df)} records")

# Clean salary columns
def clean_salary(val):
    if pd.isna(val):
        return 0
    if isinstance(val, str):
        val = re.sub(r'[^\d.]', '', val)
    return float(val) if val else 0

for col in salary_df.columns:
    if 'salary' in col.lower():
        salary_df[col] = salary_df[col].apply(clean_salary)

# Load Time to Correct
time_df = pd.read_csv(os.path.join(resources_path, 'Time_to_Correct.csv'))
print(f"✅ Time to Correct: {len(time_df)} records")

# Load Role Groups
role_groups_df = pd.read_csv(os.path.join(resources_path, 'Role_Groups.csv'))
print(f"✅ Role Groups: {len(role_groups_df)} records")

# Load classifications and job descriptions
classifications = pd.read_csv(os.path.join(inputs_path, 'Job_Classifications_Batch.csv'))
job_descriptions = pd.read_csv(os.path.join(inputs_path, 'Sample_JDs.csv'))

print(f"\n📊 Data Summary:")
print(f"  Classifications: {len(classifications)} records")
print(f"  Job Descriptions: {len(job_descriptions)} records")

# Merge classifications with job descriptions
if 'source_row_index' in classifications.columns:
    merged_data = classifications.merge(
        job_descriptions,
        left_on='source_row_index',
        right_index=True,
        how='left',
        suffixes=('', '_jd')
    )
else:
    # Try to merge on job title or ID
    merge_col = None
    for col in ['Job Title', 'job_title', 'Position Title', 'Title']:
        if col in classifications.columns and col in job_descriptions.columns:
            merge_col = col
            break
    
    if merge_col:
        merged_data = classifications.merge(
            job_descriptions,
            on=merge_col,
            how='left',
            suffixes=('', '_jd')
        )
    else:
        merged_data = classifications.copy()
        print("⚠️ Could not merge with job descriptions - proceeding with classifications only")

print(f"\n✅ Merged data: {len(merged_data)} records")
print(f"\nColumns available: {', '.join(merged_data.columns[:10])}...")

In [ ]:
#@title 5️⃣ Build Dual KSAC Analysis Engines (v4 TF-IDF + v7 Embeddings) { display-mode: "form" }

print("🔬 Building KSAC analysis engines...\n")

# ========== 1. Build KSAC Embedding Engine (v7 approach) ==========
print("1️⃣ Building Embedding-based KSAC Engine...")

# Process KSAC data for embeddings
ksac_df['Role'] = ksac_df['Role'].str.strip() if 'Role' in ksac_df.columns else ksac_df.iloc[:, 0].str.strip()

# Check for KSAC dimensions or text
if 'KSAC_Text' in ksac_df.columns:
    # Text-based KSACs - need to create embeddings
    print("  📝 Text-based KSACs detected")
    use_text_ksacs = True
else:
    # Dimension-based KSACs
    print("  📊 Dimension-based KSACs detected")
    use_text_ksacs = False
    ksac_dimensions = [col for col in ksac_df.columns if col != 'Role']
    print(f"  Dimensions: {', '.join(ksac_dimensions[:5])}...")

# ========== 2. Build TF-IDF Engine (v4 approach) ==========
if ENABLE_TFIDF_ANALYSIS and use_text_ksacs:
    print("\n2️⃣ Building TF-IDF KSAC Engine...")
    
    # Create role-to-KSAC text mapping
    ksac_df['KSAC_Text'] = ksac_df['KSAC_Text'].fillna('').str.lower()
    role_ksac_map = ksac_df.groupby('Role')['KSAC_Text'].apply(' '.join).to_dict()
    all_roles = list(role_ksac_map.keys())
    
    # Fit TF-IDF vectorizer
    tfidf = TfidfVectorizer(
        stop_words='english',
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=500,
        min_df=1,
        max_df=0.9
    )
    
    ksac_corpus = [role_ksac_map[r] for r in all_roles]
    tfidf_matrix = tfidf.fit_transform(ksac_corpus)
    role_to_idx = {role: i for i, role in enumerate(all_roles)}
    
    print(f"  ✅ TF-IDF matrix shape: {tfidf_matrix.shape}")
    print(f"  ✅ Vocabulary size: {len(tfidf.vocabulary_)}")
    
    # Store top features for each role
    feature_names = tfidf.get_feature_names_out()
    role_top_features = {}
    for role, idx in role_to_idx.items():
        role_vec = tfidf_matrix[idx].toarray().flatten()
        top_indices = role_vec.argsort()[-10:][::-1]
        role_top_features[role] = [feature_names[i] for i in top_indices]
    
    print("\n  Sample role features:")
    for role in list(role_top_features.keys())[:3]:
        print(f"    {role}: {', '.join(role_top_features[role][:5])}")
else:
    print("\n2️⃣ TF-IDF Engine: Skipped (using dimension-based KSACs)")
    tfidf = None
    tfidf_matrix = None
    role_to_idx = None

# ========== 3. Create similarity functions ==========
def calculate_embedding_similarity(role1, role2):
    """Calculate similarity using KSAC embeddings"""
    if use_text_ksacs:
        # For text-based, use cosine similarity of TF-IDF vectors
        if role1 in role_to_idx and role2 in role_to_idx:
            vec1 = tfidf_matrix[role_to_idx[role1]]
            vec2 = tfidf_matrix[role_to_idx[role2]]
            return cosine_similarity(vec1, vec2)[0][0]
    else:
        # For dimension-based, use cosine similarity of dimension vectors
        if role1 in ksac_df['Role'].values and role2 in ksac_df['Role'].values:
            vec1 = ksac_df[ksac_df['Role'] == role1][ksac_dimensions].values.flatten()
            vec2 = ksac_df[ksac_df['Role'] == role2][ksac_dimensions].values.flatten()
            return 1 - cosine(vec1, vec2)
    return 0.5  # Default similarity

def calculate_tfidf_similarity(text1, text2):
    """Calculate similarity between two text descriptions using TF-IDF"""
    if tfidf is not None:
        vecs = tfidf.transform([text1.lower(), text2.lower()])
        return cosine_similarity(vecs[0], vecs[1])[0][0]
    return 0.5

def get_ensemble_similarity(role1, role2, text1=None, text2=None):
    """Get ensemble similarity combining both methods"""
    embed_sim = calculate_embedding_similarity(role1, role2)
    
    if KSAC_METHOD == "TF-IDF" and text1 and text2:
        return calculate_tfidf_similarity(text1, text2)
    elif KSAC_METHOD == "Both (Ensemble)" and text1 and text2:
        tfidf_sim = calculate_tfidf_similarity(text1, text2)
        return 0.6 * embed_sim + 0.4 * tfidf_sim
    else:
        return embed_sim

print("\n✅ KSAC analysis engines ready!")
print(f"  Method: {KSAC_METHOD}")
print(f"  Roles available: {len(ksac_df['Role'].unique())}")

In [ ]:
#@title 6️⃣ Comprehensive Scoring and Analysis Functions { display-mode: "form" }

def find_alternative_roles(assigned_role, top_n=3):
    """Find most similar alternative roles"""
    similarities = []
    unique_roles = ksac_df['Role'].unique()
    
    for role in unique_roles:
        if role != assigned_role:
            sim = calculate_embedding_similarity(assigned_role, role)
            similarities.append((role, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

def calculate_error_severity(assigned_role, alternatives):
    """Calculate error severity based on KSAC dissimilarity"""
    if not alternatives:
        return 0.5
    
    best_similarity = alternatives[0][1] if alternatives else 0.5
    severity = 1.0 - best_similarity
    return severity

def calculate_salary_cost(role, subgroup, error_severity):
    """Calculate financial impact of misclassification"""
    # Find salary for role
    salary = 60000  # Default
    
    # Try exact match first
    salary_match = salary_df[salary_df['Major Role Grouping'].str.strip() == role]
    if not salary_match.empty:
        if 'Average Annual Salary' in salary_match.columns:
            salary = salary_match['Average Annual Salary'].iloc[0]
        elif 'Base_Salary' in salary_match.columns:
            salary = salary_match['Base_Salary'].iloc[0]
    
    # Apply subgroup multiplier
    subgroup_mult = {'I': 0.85, 'II': 1.0, 'III': 1.2}.get(subgroup, 1.0)
    adjusted_salary = salary * subgroup_mult
    
    # Calculate cost
    productivity_loss = error_severity * 0.25
    tier_mult = 2.0 if adjusted_salary >= 100000 else 1.5 if adjusted_salary >= 70000 else 1.0
    annual_cost = adjusted_salary * productivity_loss * tier_mult
    
    return annual_cost, adjusted_salary

def estimate_correction_time(error_severity, role):
    """Estimate hours to correct misclassification"""
    # Base time from data or default
    if not time_df.empty and 'Average' in time_df.columns:
        base = time_df['Average'].iloc[0]
    else:
        base = BASE_CORRECTION_TIME
    
    # Role complexity multiplier
    complexity = {
        'Director': 2.0, 'Manager': 1.8, 'Specialist': 1.5,
        'Analyst': 1.4, 'Counselor': 1.4, 'Coach': 1.3,
        'Teacher': 1.3, 'Coordinator': 1.2, 'Accountant': 1.2,
        'Technician': 1.1, 'Instructor': 1.0, 'Assistant': 0.8
    }.get(role, 1.0)
    
    # Severity multiplier
    if error_severity >= CRITICAL_ERROR_THRESHOLD:
        severity_mult = CRITICAL_TIME_MULTIPLIER
    elif error_severity >= MAJOR_ERROR_THRESHOLD:
        severity_mult = MAJOR_TIME_MULTIPLIER
    elif error_severity >= MINOR_ERROR_THRESHOLD:
        severity_mult = MINOR_TIME_MULTIPLIER
    else:
        severity_mult = 1.0
    
    total_time = base * complexity * severity_mult * 1.25  # 25% admin overhead
    return total_time

def calculate_enhanced_confidence(row, error_severity):
    """Calculate 6-component confidence score"""
    components = {}
    
    # Get relevant fields
    justification = str(row.get('grouping_justification', ''))
    job_desc = str(row.get('Position Summary', ''))
    classification = row.get('major_role_group', '')
    subgroup = row.get('sub_role_group', '')
    
    # 1. Justification quality (0-0.25)
    just_len = len(justification)
    if just_len > 500 and any(phrase in justification.lower() for phrase in 
                              ['aligns with', 'demonstrates', 'clearly shows', 'specifically']):
        components['justification'] = 0.25
    elif just_len > 200:
        components['justification'] = 0.15
    else:
        components['justification'] = 0.05
    
    # 2. KSAC alignment (0-0.25)
    components['ksac_alignment'] = (1.0 - error_severity) * 0.25
    
    # 3. Role keyword matching (0-0.15)
    role_keywords = {
        'Manager': ['manage', 'supervise', 'lead', 'oversee', 'budget', 'team'],
        'Coordinator': ['coordinate', 'organize', 'facilitate', 'arrange', 'schedule'],
        'Analyst': ['analyze', 'data', 'research', 'evaluate', 'report', 'metrics'],
        'Teacher': ['teach', 'instruct', 'curriculum', 'classroom', 'students', 'lesson'],
        'Specialist': ['specialized', 'expert', 'technical', 'consultation', 'advisory'],
        'Technician': ['diagnose', 'repair', 'maintain', 'troubleshoot', 'equipment'],
        'Coach': ['coach', 'mentor', 'develop', 'observe', 'feedback', 'model']
    }
    
    if classification in role_keywords:
        matches = sum(1 for kw in role_keywords[classification] 
                     if kw in job_desc.lower() or kw in justification.lower())
        components['role_match'] = min(0.15, matches * 0.03)
    else:
        components['role_match'] = 0.075
    
    # 4. Subgroup clarity (0-0.15)
    subgroup_indicators = {
        'I': ['entry', 'basic', 'assists', 'under supervision', 'learning'],
        'II': ['intermediate', 'independent', 'moderate', 'coordinates'],
        'III': ['senior', 'expert', 'leads', 'strategic', 'complex']
    }
    
    if subgroup in subgroup_indicators:
        matches = sum(1 for ind in subgroup_indicators[subgroup] 
                     if ind in job_desc.lower())
        components['subgroup_clarity'] = min(0.15, 0.05 + matches * 0.03)
    else:
        components['subgroup_clarity'] = 0.075
    
    # 5. Consistency check (0-0.10)
    conflicting_keywords = 0
    for other_role, keywords in role_keywords.items():
        if other_role != classification:
            matches = sum(1 for kw in keywords[:3] if kw in job_desc.lower())
            if matches >= 2:
                conflicting_keywords += 1
    
    if conflicting_keywords == 0:
        components['consistency'] = 0.10
    elif conflicting_keywords == 1:
        components['consistency'] = 0.05
    else:
        components['consistency'] = 0.02
    
    # 6. Completeness (0-0.10)
    required_fields = ['major_role_group', 'grouping_justification']
    optional_fields = ['sub_role_group', 'Position Summary']
    
    req_complete = sum(1 for f in required_fields if row.get(f) and str(row.get(f)).strip())
    opt_complete = sum(1 for f in optional_fields if row.get(f) and str(row.get(f)).strip())
    
    completeness_score = (req_complete / len(required_fields)) * 0.07 + \
                        (opt_complete / len(optional_fields)) * 0.03
    components['completeness'] = completeness_score
    
    # Calculate total confidence
    total_confidence = sum(components.values())
    
    return total_confidence, components

def detect_borderline_and_suggest(row, confidence, error_severity, alternatives):
    """Enhanced borderline detection with JD suggestions"""
    is_borderline = confidence < CONFIDENCE_THRESHOLD or error_severity > MAJOR_ERROR_THRESHOLD
    
    suggestions = []
    alternative_role = None
    
    if is_borderline:
        if alternatives and alternatives[0][1] > 0.85:
            alternative_role = alternatives[0][0]
            suggestions.append(f"Consider {alternative_role} (similarity: {alternatives[0][1]:.2f})")
        
        if confidence < 0.4:
            suggestions.append("Add more specific role indicators to JD")
        
        if error_severity > 0.7:
            suggestions.append("Review KSAC alignment carefully")
    
    return is_borderline, alternative_role, '; '.join(suggestions)

def calculate_likelihood_score(accuracy_equivalent):
    """Convert accuracy to 0-5 likelihood score"""
    if accuracy_equivalent >= HUMAN_BASELINE_MAX:
        return 5.0
    elif accuracy_equivalent >= HUMAN_BASELINE_TYPICAL:
        return 4.0 + (accuracy_equivalent - HUMAN_BASELINE_TYPICAL) / (HUMAN_BASELINE_MAX - HUMAN_BASELINE_TYPICAL)
    elif accuracy_equivalent >= HUMAN_BASELINE_MIN:
        return 3.0 + (accuracy_equivalent - HUMAN_BASELINE_MIN) / (HUMAN_BASELINE_TYPICAL - HUMAN_BASELINE_MIN)
    elif accuracy_equivalent >= 70:
        return 2.0 + (accuracy_equivalent - 70) / (HUMAN_BASELINE_MIN - 70)
    elif accuracy_equivalent >= 50:
        return 1.0 + (accuracy_equivalent - 50) / 20
    else:
        return accuracy_equivalent / 50

print("✅ Comprehensive scoring functions loaded!")
print(f"\nFeatures active:")
print(f"  • Enhanced confidence with 6 components")
print(f"  • KSAC similarity analysis ({KSAC_METHOD})")
print(f"  • Financial impact calculation")
print(f"  • Borderline detection" if ENABLE_BORDERLINE_DETECTION else "")
print(f"  • JD improvement suggestions" if ENABLE_JD_SUGGESTIONS else "")

In [ ]:
#@title 7️⃣ Run Complete Analysis on All Records { display-mode: "form" }

print("🚀 Running comprehensive analysis...\n")
print(f"Processing {len(merged_data)} records...")
print("="*60)

results = []
pattern_tracker = {}  # For pattern detection

for idx, row in merged_data.iterrows():
    if idx % 50 == 0:
        print(f"  Processing record {idx+1}/{len(merged_data)}...")
    
    result = {
        'index': idx,
        'job_title': row.get('Job Title', row.get('job_title', '')),
        'classification': row.get('major_role_group', ''),
        'subgroup': row.get('sub_role_group', ''),
        'justification': row.get('grouping_justification', '')
    }
    
    # Find alternative roles
    alternatives = find_alternative_roles(result['classification'])
    result['alternatives'] = alternatives
    result['best_alternative'] = alternatives[0][0] if alternatives else None
    result['best_alt_similarity'] = alternatives[0][1] if alternatives else 0.5
    
    # Calculate error severity
    error_severity = calculate_error_severity(result['classification'], alternatives)
    result['error_severity'] = error_severity
    
    # Calculate salary cost
    annual_cost, salary = calculate_salary_cost(
        result['classification'],
        result['subgroup'],
        error_severity
    )
    result['salary_cost_annual'] = annual_cost
    result['salary_amount'] = salary
    
    # Estimate correction time
    correction_hours = estimate_correction_time(error_severity, result['classification'])
    result['correction_time_hours'] = correction_hours
    
    # Calculate confidence
    confidence, conf_components = calculate_enhanced_confidence(row, error_severity)
    result['confidence_score'] = confidence
    result['confidence_components'] = conf_components
    
    # Borderline detection
    if ENABLE_BORDERLINE_DETECTION:
        is_borderline, alt_suggestion, jd_suggestions = detect_borderline_and_suggest(
            row, confidence, error_severity, alternatives
        )
        result['is_borderline'] = is_borderline
        result['alternative_suggestion'] = alt_suggestion
        result['jd_suggestions'] = jd_suggestions if ENABLE_JD_SUGGESTIONS else ''
    else:
        result['is_borderline'] = False
        result['alternative_suggestion'] = None
        result['jd_suggestions'] = ''
    
    # Calculate accuracy equivalent
    weighted_score = (
        (1 - error_severity) * KSAC_DISSIMILARITY_WEIGHT +
        (1 - annual_cost/100000) * SALARY_IMPACT_WEIGHT +
        (1 - correction_hours/100) * CORRECTION_TIME_WEIGHT +
        confidence * 0.2
    )
    accuracy_equivalent = min(100, max(0, weighted_score * 100))
    result['accuracy_equivalent'] = accuracy_equivalent
    
    # Calculate likelihood score
    likelihood = calculate_likelihood_score(accuracy_equivalent)
    result['likelihood_score'] = likelihood
    
    # Performance category
    if accuracy_equivalent >= HUMAN_BASELINE_MAX:
        result['performance_category'] = 'Exceeds Human'
    elif accuracy_equivalent >= HUMAN_BASELINE_MIN:
        result['performance_category'] = 'Human-Level'
    elif accuracy_equivalent >= 75:
        result['performance_category'] = 'Near-Human'
    elif accuracy_equivalent >= 60:
        result['performance_category'] = 'Acceptable'
    else:
        result['performance_category'] = 'Needs Review'
    
    # Severity category
    if error_severity >= CRITICAL_ERROR_THRESHOLD:
        result['severity_category'] = 'Critical'
    elif error_severity >= MAJOR_ERROR_THRESHOLD:
        result['severity_category'] = 'Major'
    elif error_severity >= MINOR_ERROR_THRESHOLD:
        result['severity_category'] = 'Minor'
    else:
        result['severity_category'] = 'Negligible'
    
    # Priority for review
    if result['severity_category'] == 'Critical' or confidence < 0.4:
        result['review_priority'] = 'Critical'
    elif result['is_borderline'] or result['severity_category'] == 'Major':
        result['review_priority'] = 'High'
    elif confidence < CONFIDENCE_THRESHOLD:
        result['review_priority'] = 'Medium'
    else:
        result['review_priority'] = 'Low'
    
    # Pattern tracking
    if ENABLE_PATTERN_DETECTION:
        pattern_key = f"{result['classification']}_{result['subgroup']}"
        if pattern_key not in pattern_tracker:
            pattern_tracker[pattern_key] = {
                'count': 0,
                'total_confidence': 0,
                'total_severity': 0,
                'borderline_count': 0
            }
        pattern_tracker[pattern_key]['count'] += 1
        pattern_tracker[pattern_key]['total_confidence'] += confidence
        pattern_tracker[pattern_key]['total_severity'] += error_severity
        if result['is_borderline']:
            pattern_tracker[pattern_key]['borderline_count'] += 1
    
    results.append(result)

# Create results DataFrame
results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("✅ Analysis complete!")
print(f"\nProcessed {len(results_df)} records")
print(f"\nPerformance Distribution:")
print(results_df['performance_category'].value_counts())
print(f"\nAverage Metrics:")
print(f"  Likelihood Score: {results_df['likelihood_score'].mean():.2f}")
print(f"  Confidence Score: {results_df['confidence_score'].mean():.2f}")
print(f"  Error Severity: {results_df['error_severity'].mean():.2f}")
print(f"  Annual Cost: ${results_df['salary_cost_annual'].mean():,.2f}")

In [ ]:
#@title 8️⃣ Confusion Matrix Analysis (from v2 Evaluation) { display-mode: "form" }

if ENABLE_CONFUSION_ANALYSIS and INCLUDE_GROUND_TRUTH:
    print("📊 Generating Confusion Matrix Analysis...\n")
    
    # Check if we have ground truth data
    if 'true_major_group' in results_df.columns or 'ground_truth' in results_df.columns:
        
        # Identify true and predicted columns
        true_col = 'true_major_group' if 'true_major_group' in results_df.columns else 'ground_truth'
        pred_col = 'classification'
        
        # Standard confusion matrix
        all_classes = sorted(list(set(results_df[true_col].unique()) | set(results_df[pred_col].unique())))
        conf_matrix = confusion_matrix(
            results_df[true_col],
            results_df[pred_col],
            labels=all_classes
        )
        
        # Create confusion DataFrame
        confusion_df = pd.DataFrame(
            conf_matrix,
            index=all_classes,
            columns=all_classes
        )
        
        print("Standard Confusion Matrix:")
        print(confusion_df)
        
        # Severity-weighted confusion matrix
        print("\nCalculating Severity-Weighted Confusion Matrix...")
        
        severity_matrix = pd.DataFrame(
            0.0,
            index=all_classes,
            columns=all_classes
        )
        
        for idx, row in results_df.iterrows():
            true_class = row[true_col]
            pred_class = row[pred_col]
            
            # Calculate severity cost (from v2 approach)
            similarity = row.get('best_alt_similarity', 0.5)
            dissimilarity = 1 - similarity
            
            # Normalize salary and hours
            max_salary = results_df['salary_amount'].max()
            max_hours = results_df['correction_time_hours'].max()
            
            salary_norm = row['salary_amount'] / max_salary if max_salary > 0 else 0
            hours_norm = row['correction_time_hours'] / max_hours if max_hours > 0 else 0
            
            # Calculate severity cost index
            severity_cost = dissimilarity * (
                SEVERITY_SALARY_WEIGHT * salary_norm +
                SEVERITY_TIME_WEIGHT * hours_norm
            )
            
            severity_matrix.loc[true_class, pred_class] += severity_cost
        
        print("\nSeverity-Weighted Confusion Matrix:")
        print(severity_matrix.round(2))
        
        # Calculate metrics
        is_correct = results_df[true_col] == results_df[pred_col]
        accuracy = is_correct.mean()
        
        print(f"\nEvaluation Metrics:")
        print(f"  Overall Accuracy: {accuracy:.2%}")
        print(f"  Total Misclassifications: {(~is_correct).sum()}")
        print(f"  Total Severity Cost: {severity_matrix.values.sum():.2f}")
        
        # Per-class metrics
        print("\nPer-Class Accuracy:")
        for class_name in all_classes:
            class_mask = results_df[true_col] == class_name
            if class_mask.sum() > 0:
                class_acc = (results_df.loc[class_mask, true_col] == 
                           results_df.loc[class_mask, pred_col]).mean()
                print(f"  {class_name}: {class_acc:.2%} ({class_mask.sum()} samples)")
        
        # Save confusion matrices
        confusion_df.to_csv(os.path.join(run_results_path, 'confusion_matrix_standard.csv'))
        severity_matrix.to_csv(os.path.join(run_results_path, 'confusion_matrix_severity.csv'))
        print("\n✅ Confusion matrices saved!")
        
    else:
        print("⚠️ No ground truth data available for confusion matrix analysis")
        print("   To enable: Include 'true_major_group' or 'ground_truth' column in your data")
else:
    print("ℹ️ Confusion matrix analysis disabled or no ground truth available")
    print("   To enable: Set ENABLE_CONFUSION_ANALYSIS=True and INCLUDE_GROUND_TRUTH=True")

In [ ]:
#@title 9️⃣ Generate Enhanced 12-Panel Visualization Dashboard { display-mode: "form" }

print("📊 Generating comprehensive visualization dashboard...\n")

# Create figure with 12 subplots (4x3 grid)
fig = plt.figure(figsize=(20, 24))
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.3, wspace=0.25)

# Color schemes
colors = sns.color_palette("husl", 8)

# Panel 1: Likelihood Score Distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(results_df['likelihood_score'], bins=20, color=colors[0], edgecolor='black', alpha=0.7)
ax1.axvline(4.0, color='red', linestyle='--', label=f'Human Baseline ({HUMAN_BASELINE_TYPICAL}%)')
ax1.set_xlabel('Likelihood Score')
ax1.set_ylabel('Frequency')
ax1.set_title('Likelihood Score Distribution')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Panel 2: Confidence vs Error Severity
ax2 = fig.add_subplot(gs[0, 1])
scatter = ax2.scatter(results_df['error_severity'], results_df['confidence_score'],
                     c=results_df['likelihood_score'], cmap='RdYlGn', s=50, alpha=0.6)
ax2.set_xlabel('Error Severity')
ax2.set_ylabel('Confidence Score')
ax2.set_title('Confidence vs Error Severity')
ax2.axhline(CONFIDENCE_THRESHOLD, color='orange', linestyle='--', alpha=0.5, label='Confidence Threshold')
ax2.axvline(MAJOR_ERROR_THRESHOLD, color='red', linestyle='--', alpha=0.5, label='Major Error')
plt.colorbar(scatter, ax=ax2, label='Likelihood')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Panel 3: Performance Categories
ax3 = fig.add_subplot(gs[0, 2])
perf_counts = results_df['performance_category'].value_counts()
ax3.pie(perf_counts.values, labels=perf_counts.index, autopct='%1.1f%%',
        colors=colors[:len(perf_counts)], startangle=90)
ax3.set_title('Performance Category Distribution')

# Panel 4: Severity Categories by Role
ax4 = fig.add_subplot(gs[1, 0])
severity_by_role = pd.crosstab(results_df['classification'], results_df['severity_category'])
severity_by_role.plot(kind='bar', stacked=True, ax=ax4, color=['green', 'yellow', 'orange', 'red'])
ax4.set_xlabel('Role')
ax4.set_ylabel('Count')
ax4.set_title('Severity Categories by Role')
ax4.legend(title='Severity', bbox_to_anchor=(1.05, 1), loc='upper left')
ax4.tick_params(axis='x', rotation=45)

# Panel 5: Cost Analysis
ax5 = fig.add_subplot(gs[1, 1])
top_costs = results_df.nlargest(10, 'salary_cost_annual')
ax5.barh(range(len(top_costs)), top_costs['salary_cost_annual'], color=colors[3])
ax5.set_yticks(range(len(top_costs)))
ax5.set_yticklabels([f"{r['job_title'][:20]}..." for _, r in top_costs.iterrows()])
ax5.set_xlabel('Annual Cost ($)')
ax5.set_title('Top 10 Highest Cost Misclassifications')
ax5.grid(True, alpha=0.3)

# Panel 6: Borderline Cases
ax6 = fig.add_subplot(gs[1, 2])
borderline_df = results_df[results_df['is_borderline']]
if len(borderline_df) > 0:
    borderline_by_role = borderline_df['classification'].value_counts()[:8]
    ax6.bar(range(len(borderline_by_role)), borderline_by_role.values, color=colors[5])
    ax6.set_xticks(range(len(borderline_by_role)))
    ax6.set_xticklabels(borderline_by_role.index, rotation=45, ha='right')
    ax6.set_ylabel('Count')
    ax6.set_title(f'Borderline Cases by Role (n={len(borderline_df)})')
else:
    ax6.text(0.5, 0.5, 'No borderline cases', ha='center', va='center')
    ax6.set_title('Borderline Cases')
ax6.grid(True, alpha=0.3)

# Panel 7: Review Priority Distribution
ax7 = fig.add_subplot(gs[2, 0])
priority_counts = results_df['review_priority'].value_counts()
priority_colors = {'Critical': 'red', 'High': 'orange', 'Medium': 'yellow', 'Low': 'green'}
ax7.bar(priority_counts.index, priority_counts.values,
       color=[priority_colors.get(p, 'gray') for p in priority_counts.index])
ax7.set_xlabel('Priority Level')
ax7.set_ylabel('Count')
ax7.set_title('Review Priority Distribution')
ax7.grid(True, alpha=0.3)

# Panel 8: KSAC Similarity Distribution
ax8 = fig.add_subplot(gs[2, 1])
ax8.hist(results_df['best_alt_similarity'], bins=20, color=colors[6], edgecolor='black', alpha=0.7)
ax8.axvline(0.85, color='green', linestyle='--', label='Strong Match (>0.85)')
ax8.axvline(0.70, color='orange', linestyle='--', label='Weak Match (<0.70)')
ax8.set_xlabel('KSAC Similarity Score')
ax8.set_ylabel('Frequency')
ax8.set_title('KSAC Similarity to Best Alternative')
ax8.legend()
ax8.grid(True, alpha=0.3)

# Panel 9: Correction Time Distribution
ax9 = fig.add_subplot(gs[2, 2])
ax9.hist(results_df['correction_time_hours'], bins=20, color=colors[7], edgecolor='black', alpha=0.7)
ax9.axvline(results_df['correction_time_hours'].mean(), color='red', linestyle='--',
           label=f'Mean: {results_df["correction_time_hours"].mean():.1f}h')
ax9.set_xlabel('Correction Time (hours)')
ax9.set_ylabel('Frequency')
ax9.set_title('Estimated Correction Time Distribution')
ax9.legend()
ax9.grid(True, alpha=0.3)

# Panel 10: Confidence Components Breakdown (if patterns detected)
ax10 = fig.add_subplot(gs[3, 0])
if ENABLE_PATTERN_DETECTION and pattern_tracker:
    # Calculate average confidence by component
    comp_names = list(results_df.iloc[0]['confidence_components'].keys())
    comp_avgs = {comp: np.mean([r['confidence_components'][comp] for r in results]) 
                for comp in comp_names}
    
    ax10.bar(comp_names, comp_avgs.values(), color=colors[1])
    ax10.set_xlabel('Component')
    ax10.set_ylabel('Average Score')
    ax10.set_title('Average Confidence Component Scores')
    ax10.tick_params(axis='x', rotation=45)
else:
    ax10.text(0.5, 0.5, 'Pattern detection disabled', ha='center', va='center')
    ax10.set_title('Pattern Analysis')
ax10.grid(True, alpha=0.3)

# Panel 11: Accuracy Equivalent vs Human Baseline
ax11 = fig.add_subplot(gs[3, 1])
ax11.hist(results_df['accuracy_equivalent'], bins=20, color=colors[2], edgecolor='black', alpha=0.7)
ax11.axvspan(HUMAN_BASELINE_MIN, HUMAN_BASELINE_MAX, alpha=0.3, color='green', label='Human Range')
ax11.axvline(HUMAN_BASELINE_TYPICAL, color='darkgreen', linestyle='-', linewidth=2, label='Human Typical')
ax11.set_xlabel('Accuracy Equivalent (%)')
ax11.set_ylabel('Frequency')
ax11.set_title('Accuracy Equivalent Distribution')
ax11.legend()
ax11.grid(True, alpha=0.3)

# Panel 12: Summary Statistics Box
ax12 = fig.add_subplot(gs[3, 2])
ax12.axis('off')

summary_text = f"""SUMMARY STATISTICS
═══════════════════════════════
Total Records: {len(results_df)}
Avg Likelihood: {results_df['likelihood_score'].mean():.2f}
Avg Confidence: {results_df['confidence_score'].mean():.2f}
Avg Error Severity: {results_df['error_severity'].mean():.2f}

PERFORMANCE
─────────────────────
Human-Level+: {((results_df['accuracy_equivalent'] >= HUMAN_BASELINE_MIN).sum() / len(results_df) * 100):.1f}%
Borderline: {(results_df['is_borderline'].sum() / len(results_df) * 100):.1f}%
Critical Priority: {((results_df['review_priority'] == 'Critical').sum())} records

FINANCIAL IMPACT
─────────────────────
Total Annual Cost: ${results_df['salary_cost_annual'].sum():,.0f}
Avg Cost/Error: ${results_df['salary_cost_annual'].mean():,.0f}
Total Correction Hours: {results_df['correction_time_hours'].sum():,.0f}

TOP ISSUES
─────────────────────
"""

# Add top problematic roles
problem_roles = results_df[results_df['error_severity'] > MAJOR_ERROR_THRESHOLD]['classification'].value_counts()[:3]
for role, count in problem_roles.items():
    summary_text += f"{role}: {count} errors\n"

ax12.text(0.1, 0.95, summary_text, transform=ax12.transAxes, fontsize=10,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle(f'MNPS Job Classification Analysis Dashboard v8.0 - {timestamp}', fontsize=16, y=1.0)

# Save visualization
viz_path = os.path.join(run_results_path, 'dashboard_12panel_v8.png')
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ 12-panel dashboard saved to: {viz_path}")

In [ ]:
#@title 10️⃣ Save All Results and Generate Reports { display-mode: "form" }

print("💾 Saving all results and generating reports...\n")

# 1. Save main results DataFrame
results_path_csv = os.path.join(run_results_path, 'complete_analysis_v8.csv')
results_df.to_csv(results_path_csv, index=False)
print(f"✅ Main results saved: complete_analysis_v8.csv")

# 2. Save Excel report with multiple sheets
excel_path = os.path.join(run_results_path, 'comprehensive_report_v8.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Main results
    results_df.to_excel(writer, sheet_name='Complete Analysis', index=False)
    
    # Summary statistics
    summary_stats = pd.DataFrame({
        'Metric': ['Total Records', 'Avg Likelihood Score', 'Avg Confidence', 
                  'Avg Error Severity', 'Total Annual Cost', 'Total Correction Hours',
                  'Human-Level Performance %', 'Borderline Cases', 'Critical Priority Cases'],
        'Value': [len(results_df), results_df['likelihood_score'].mean(),
                 results_df['confidence_score'].mean(), results_df['error_severity'].mean(),
                 results_df['salary_cost_annual'].sum(), results_df['correction_time_hours'].sum(),
                 (results_df['accuracy_equivalent'] >= HUMAN_BASELINE_MIN).mean() * 100,
                 results_df['is_borderline'].sum(),
                 (results_df['review_priority'] == 'Critical').sum()]
    })
    summary_stats.to_excel(writer, sheet_name='Summary', index=False)
    
    # Critical errors
    critical_df = results_df[results_df['review_priority'] == 'Critical']
    if len(critical_df) > 0:
        critical_df.to_excel(writer, sheet_name='Critical Cases', index=False)
    
    # Borderline cases
    if len(borderline_df) > 0:
        borderline_df.to_excel(writer, sheet_name='Borderline Cases', index=False)
    
    # Top cost errors
    top_cost = results_df.nlargest(20, 'salary_cost_annual')
    top_cost.to_excel(writer, sheet_name='Top Cost Errors', index=False)
    
    # Pattern analysis
    if ENABLE_PATTERN_DETECTION and pattern_tracker:
        pattern_df = pd.DataFrame.from_dict(pattern_tracker, orient='index')
        pattern_df['avg_confidence'] = pattern_df['total_confidence'] / pattern_df['count']
        pattern_df['avg_severity'] = pattern_df['total_severity'] / pattern_df['count']
        pattern_df['borderline_rate'] = pattern_df['borderline_count'] / pattern_df['count']
        pattern_df.to_excel(writer, sheet_name='Pattern Analysis')

print(f"✅ Excel report saved: comprehensive_report_v8.xlsx")

# 3. Generate text report
report_path = os.path.join(run_results_path, 'executive_summary_v8.txt')
with open(report_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("MNPS JOB CLASSIFICATION ANALYSIS - EXECUTIVE SUMMARY\n")
    f.write(f"Version 8.0 Ultimate Enhanced | Generated: {timestamp}\n")
    f.write("="*80 + "\n\n")
    
    f.write("OVERALL PERFORMANCE\n")
    f.write("-"*40 + "\n")
    f.write(f"Total Classifications Analyzed: {len(results_df)}\n")
    f.write(f"Average Likelihood Score: {results_df['likelihood_score'].mean():.2f}/5.00\n")
    f.write(f"Human-Level Performance: {(results_df['accuracy_equivalent'] >= HUMAN_BASELINE_MIN).mean()*100:.1f}%\n")
    f.write(f"Exceeds Human Performance: {(results_df['accuracy_equivalent'] >= HUMAN_BASELINE_MAX).mean()*100:.1f}%\n")
    f.write("\n")
    
    f.write("CONFIDENCE ANALYSIS\n")
    f.write("-"*40 + "\n")
    f.write(f"Average Confidence Score: {results_df['confidence_score'].mean():.3f}\n")
    f.write(f"High Confidence (≥{CONFIDENCE_HIGH}): {(results_df['confidence_score'] >= CONFIDENCE_HIGH).sum()} records\n")
    f.write(f"Low Confidence (<{CONFIDENCE_THRESHOLD}): {(results_df['confidence_score'] < CONFIDENCE_THRESHOLD).sum()} records\n")
    f.write("\n")
    
    f.write("FINANCIAL IMPACT\n")
    f.write("-"*40 + "\n")
    f.write(f"Total Annual Error Cost: ${results_df['salary_cost_annual'].sum():,.0f}\n")
    f.write(f"Average Cost per Error: ${results_df['salary_cost_annual'].mean():,.0f}\n")
    f.write(f"Total Correction Hours: {results_df['correction_time_hours'].sum():,.0f}\n")
    f.write(f"Average Hours per Correction: {results_df['correction_time_hours'].mean():.1f}\n")
    f.write("\n")
    
    f.write("PRIORITY ACTIONS\n")
    f.write("-"*40 + "\n")
    f.write(f"Critical Priority Cases: {(results_df['review_priority'] == 'Critical').sum()}\n")
    f.write(f"High Priority Cases: {(results_df['review_priority'] == 'High').sum()}\n")
    f.write(f"Borderline Cases: {results_df['is_borderline'].sum()}\n")
    f.write("\n")
    
    if ENABLE_CONFUSION_ANALYSIS and 'true_major_group' in results_df.columns:
        f.write("EVALUATION METRICS\n")
        f.write("-"*40 + "\n")
        accuracy = (results_df['true_major_group'] == results_df['classification']).mean()
        f.write(f"Classification Accuracy: {accuracy:.2%}\n")
        f.write(f"Misclassification Rate: {1-accuracy:.2%}\n")
        f.write("\n")
    
    f.write("RECOMMENDATIONS\n")
    f.write("="*80 + "\n")
    f.write("1. Review all Critical Priority cases immediately\n")
    f.write("2. Focus on roles with highest error severity\n")
    f.write("3. Implement JD improvements for borderline cases\n")
    f.write("4. Consider retraining for consistently problematic classifications\n")

print(f"✅ Executive summary saved: executive_summary_v8.txt")

# 4. Save configuration
config = {
    'version': '8.0',
    'timestamp': timestamp,
    'parameters': {
        'human_baseline': {'min': HUMAN_BASELINE_MIN, 'typical': HUMAN_BASELINE_TYPICAL, 'max': HUMAN_BASELINE_MAX},
        'weights': {
            'ksac_dissimilarity': KSAC_DISSIMILARITY_WEIGHT,
            'salary_impact': SALARY_IMPACT_WEIGHT,
            'correction_time': CORRECTION_TIME_WEIGHT
        },
        'thresholds': {
            'confidence': CONFIDENCE_THRESHOLD,
            'confidence_high': CONFIDENCE_HIGH,
            'critical_error': CRITICAL_ERROR_THRESHOLD,
            'major_error': MAJOR_ERROR_THRESHOLD,
            'minor_error': MINOR_ERROR_THRESHOLD
        },
        'features': {
            'ksac_method': KSAC_METHOD,
            'borderline_detection': ENABLE_BORDERLINE_DETECTION,
            'jd_suggestions': ENABLE_JD_SUGGESTIONS,
            'pattern_detection': ENABLE_PATTERN_DETECTION,
            'dynamic_calibration': ENABLE_DYNAMIC_CALIBRATION,
            'confusion_analysis': ENABLE_CONFUSION_ANALYSIS,
            'tfidf_analysis': ENABLE_TFIDF_ANALYSIS
        }
    },
    'data_stats': {
        'total_records': len(results_df),
        'unique_roles': len(results_df['classification'].unique()),
        'borderline_cases': int(results_df['is_borderline'].sum()),
        'critical_cases': int((results_df['review_priority'] == 'Critical').sum())
    }
}

config_path = os.path.join(run_results_path, 'configuration_v8.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Configuration saved: configuration_v8.json")

print("\n" + "="*60)
print("🎉 ALL RESULTS SAVED SUCCESSFULLY!")
print("="*60)
print(f"\nResults location: {run_results_path}")
print(f"\nFiles generated:")
print(f"  • complete_analysis_v8.csv")
print(f"  • comprehensive_report_v8.xlsx")
print(f"  • executive_summary_v8.txt")
print(f"  • dashboard_12panel_v8.png")
print(f"  • configuration_v8.json")
if ENABLE_CONFUSION_ANALYSIS:
    print(f"  • confusion_matrix_standard.csv")
    print(f"  • confusion_matrix_severity.csv")

print(f"\n🚀 Analysis complete! Ready for review and decision-making.")